# Data Onboarding Workflow

This notebook converts the data onboarding workflow into executable steps for discovering, validating, cleaning, transforming, splitting, and saving datasets.

Update the configuration in the first code cell to match the paths, filenames, column names, and split settings from `implementations/CTC/data_onboard`.

## 1. Load Configuration and Define Paths

In [5]:
from pathlib import Path
from datetime import datetime, timezone
import json
import logging
import random
import re

import numpy as np
import pandas as pd

REPO_ROOT = Path("/home/coder/agentic-forecasting")
ACTIVE_WORKFLOW = REPO_ROOT / "implementations" / "CTC" / "data_onboard"

# Replace these values with the corresponding configuration from data_onboard.
INPUT_ROOT = REPO_ROOT / "data"
OUTPUT_ROOT = REPO_ROOT / "data" / "processed" / "CTC"

DATASET_NAME = "CTC"
FILE_PATTERNS = ["*.csv"]

ENTITY_COLUMN = "entity_id"
TIMESTAMP_COLUMN = "timestamp"
TARGET_COLUMN = None

REQUIRED_COLUMNS = []
CATEGORICAL_COLUMNS = []
NUMERIC_COLUMNS = []
BOOLEAN_COLUMNS = []

TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15
RANDOM_SEED = 42

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "repository_root": str(REPO_ROOT),
    "active_workflow": str(ACTIVE_WORKFLOW),
    "input_root": str(INPUT_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "dataset_name": DATASET_NAME,
    "file_patterns": FILE_PATTERNS,
    "entity_column": ENTITY_COLUMN,
    "timestamp_column": TIMESTAMP_COLUMN,
    "target_column": TARGET_COLUMN,
    "required_columns": REQUIRED_COLUMNS,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "numeric_columns": NUMERIC_COLUMNS,
    "boolean_columns": BOOLEAN_COLUMNS,
    "train_fraction": TRAIN_FRACTION,
    "validation_fraction": VALIDATION_FRACTION,
    "test_fraction": TEST_FRACTION,
    "random_seed": RANDOM_SEED,
}

pd.Series(CONFIG, name="active_configuration")

repository_root                          /home/coder/agentic-forecasting
active_workflow        /home/coder/agentic-forecasting/implementation...
input_root                          /home/coder/agentic-forecasting/data
output_root            /home/coder/agentic-forecasting/data/processed...
dataset_name                                                         CTC
file_patterns                                                    [*.csv]
entity_column                                                  entity_id
timestamp_column                                               timestamp
target_column                                                       None
required_columns                                                      []
categorical_columns                                                   []
numeric_columns                                                       []
boolean_columns                                                       []
train_fraction                                     

## 2. Import Dependencies

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
)

logger = logging.getLogger("data_onboarding")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Dependencies loaded.")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

Dependencies loaded.
pandas: 3.0.3
numpy: 2.4.6


In [ ]:
# Optional: run this cell only if the environment is missing dependencies.
# %pip install pandas numpy pyarrow

## 3. Discover and Read Input Data

In [7]:
discovered_files = []

for pattern in FILE_PATTERNS:
    discovered_files.extend(INPUT_ROOT.rglob(pattern))

discovered_files = sorted(set(discovered_files))


if not discovered_files:
    raise FileNotFoundError(
        f"No input files were found below {INPUT_ROOT}. "
        "Populate the data cache or update INPUT_ROOT."
    )

file_inventory = pd.DataFrame(
    {
        "path": [str(path) for path in discovered_files],
        "name": [path.name for path in discovered_files],
        "suffix": [path.suffix.lower() for path in discovered_files],
        "size_bytes": [path.stat().st_size for path in discovered_files],
    }
)

print(f"Discovered {len(discovered_files)} input file(s).")
display(file_inventory)

FileNotFoundError: No input files were found below /home/coder/agentic-forecasting/data. Populate the data cache or update INPUT_ROOT.

In [ ]:
def read_input_file(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)

    if suffix == ".parquet":
        return pd.read_parquet(path)

    if suffix == ".json":
        return pd.read_json(path)

    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)

    raise ValueError(f"Unsupported input format: {path}")


loaded_datasets = {}

for path in discovered_files:
    dataset_key = re.sub(r"[^A-Za-z0-9_]+", "_", path.stem).strip("_").lower()
    frame = read_input_file(path)
    loaded_datasets[dataset_key] = frame

    print(f"{path.name}: {frame.shape[0]:,} rows x {frame.shape[1]:,} columns")
    display(frame.head())

if not loaded_datasets:
    raise ValueError("No datasets were loaded.")

In [ ]:
# Combine compatible input files into one working dataset.
# If data_onboard defines separate datasets, keep the individual frames instead.
compatible_frames = list(loaded_datasets.values())
common_columns = set(compatible_frames[0].columns)

for frame in compatible_frames[1:]:
    common_columns.intersection_update(frame.columns)

if len(compatible_frames) == 1:
    data = compatible_frames[0].copy()
else:
    data = pd.concat(compatible_frames, ignore_index=True, sort=False)

print(f"Working dataset shape: {data.shape}")
display(data.head())
display(data.dtypes.rename("dtype").to_frame())

## 4. Validate File Structure and Schema

In [ ]:
validation_results = []

def record_check(check_name: str, passed: bool, details: str) -> None:
    validation_results.append(
        {
            "check": check_name,
            "passed": bool(passed),
            "details": details,
        }
    )


missing_required_columns = sorted(set(REQUIRED_COLUMNS).difference(data.columns))
record_check(
    "required_columns",
    not missing_required_columns,
    "All required columns are present."
    if not missing_required_columns
    else f"Missing columns: {missing_required_columns}",
)

record_check(
    "nonempty_dataset",
    not data.empty,
    f"{len(data):,} rows loaded.",
)

if ENTITY_COLUMN:
    record_check(
        "entity_column",
        ENTITY_COLUMN in data.columns,
        f"Entity column: {ENTITY_COLUMN}",
    )

if TIMESTAMP_COLUMN:
    record_check(
        "timestamp_column",
        TIMESTAMP_COLUMN in data.columns,
        f"Timestamp column: {TIMESTAMP_COLUMN}",
    )

validation_summary = pd.DataFrame(validation_results)
display(validation_summary)

if not validation_summary["passed"].all():
    raise ValueError("Initial schema validation failed.")

In [ ]:
# Validate configured uniqueness and value constraints when the relevant columns exist.
if ENTITY_COLUMN in data.columns and TIMESTAMP_COLUMN in data.columns:
    duplicate_key_count = data.duplicated(
        subset=[ENTITY_COLUMN, TIMESTAMP_COLUMN]
    ).sum()

    record_check(
        "unique_entity_timestamp_keys",
        duplicate_key_count == 0,
        f"{duplicate_key_count:,} duplicate keys found.",
    )

if NUMERIC_COLUMNS:
    missing_numeric_columns = sorted(set(NUMERIC_COLUMNS).difference(data.columns))
    record_check(
        "numeric_columns",
        not missing_numeric_columns,
        "All configured numeric columns are present."
        if not missing_numeric_columns
        else f"Missing numeric columns: {missing_numeric_columns}",
    )

if CATEGORICAL_COLUMNS:
    missing_categorical_columns = sorted(
        set(CATEGORICAL_COLUMNS).difference(data.columns)
    )
    record_check(
        "categorical_columns",
        not missing_categorical_columns,
        "All configured categorical columns are present."
        if not missing_categorical_columns
        else f"Missing categorical columns: {missing_categorical_columns}",
    )

if BOOLEAN_COLUMNS:
    missing_boolean_columns = sorted(set(BOOLEAN_COLUMNS).difference(data.columns))
    record_check(
        "boolean_columns",
        not missing_boolean_columns,
        "All configured Boolean columns are present."
        if not missing_boolean_columns
        else f"Missing Boolean columns: {missing_boolean_columns}",
    )

validation_summary = pd.DataFrame(validation_results)
display(validation_summary)

if not validation_summary["passed"].all():
    raise ValueError("Schema or key validation failed.")

## 5. Clean and Standardize Data

In [ ]:
def normalize_column_name(column: object) -> str:
    normalized = str(column).strip().lower()
    normalized = re.sub(r"[^a-z0-9]+", "_", normalized)
    return normalized.strip("_")


data = data.copy()
data.columns = [normalize_column_name(column) for column in data.columns]

ENTITY_COLUMN = normalize_column_name(ENTITY_COLUMN) if ENTITY_COLUMN else None
TIMESTAMP_COLUMN = (
    normalize_column_name(TIMESTAMP_COLUMN) if TIMESTAMP_COLUMN else None
)
TARGET_COLUMN = normalize_column_name(TARGET_COLUMN) if TARGET_COLUMN else None
REQUIRED_COLUMNS = [normalize_column_name(column) for column in REQUIRED_COLUMNS]
CATEGORICAL_COLUMNS = [
    normalize_column_name(column) for column in CATEGORICAL_COLUMNS
]
NUMERIC_COLUMNS = [normalize_column_name(column) for column in NUMERIC_COLUMNS]
BOOLEAN_COLUMNS = [normalize_column_name(column) for column in BOOLEAN_COLUMNS]

for column in data.select_dtypes(include=["object", "string"]).columns:
    data[column] = data[column].map(
        lambda value: value.strip() if isinstance(value, str) else value
    )

rows_before_deduplication = len(data)

deduplication_keys = [
    column
    for column in [ENTITY_COLUMN, TIMESTAMP_COLUMN]
    if column and column in data.columns
]

if deduplication_keys:
    data = data.drop_duplicates(subset=deduplication_keys, keep="last")
else:
    data = data.drop_duplicates()

rows_removed_by_deduplication = rows_before_deduplication - len(data)

print(f"Rows before deduplication: {rows_before_deduplication:,}")
print(f"Rows removed: {rows_removed_by_deduplication:,}")
print(f"Rows after deduplication: {len(data):,}")

In [ ]:
# Standardize categorical values.
for column in CATEGORICAL_COLUMNS:
    if column in data.columns:
        data[column] = (
            data[column]
            .astype("string")
            .str.strip()
            .str.lower()
            .replace({"": pd.NA, "nan": pd.NA, "none": pd.NA})
        )

if ENTITY_COLUMN in data.columns:
    data[ENTITY_COLUMN] = data[ENTITY_COLUMN].astype("string").str.strip()

if TIMESTAMP_COLUMN in data.columns:
    data = data.sort_values(
        by=[column for column in [ENTITY_COLUMN, TIMESTAMP_COLUMN] if column]
    ).reset_index(drop=True)

display(data.head())

## 6. Handle Missing Values and Invalid Records

In [ ]:
missing_profile_before = (
    data.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
    .assign(
        missing_fraction=lambda frame: frame["missing_count"] / max(len(data), 1)
    )
    .sort_values("missing_count", ascending=False)
)

display(missing_profile_before)

In [ ]:
# Convert invalid timestamp values to missing values, then quarantine those records.
invalid_timestamp_rows = 0
quarantine = pd.DataFrame(columns=data.columns)

if TIMESTAMP_COLUMN in data.columns:
    parsed_timestamps = pd.to_datetime(
        data[TIMESTAMP_COLUMN],
        errors="coerce",
        utc=True,
    )

    invalid_timestamp_mask = parsed_timestamps.isna()
    invalid_timestamp_rows = int(invalid_timestamp_mask.sum())

    quarantine = data.loc[invalid_timestamp_mask].copy()
    data.loc[:, TIMESTAMP_COLUMN] = parsed_timestamps

    data = data.loc[~invalid_timestamp_mask].copy()

required_for_valid_record = [
    column
    for column in [ENTITY_COLUMN, TIMESTAMP_COLUMN]
    if column and column in data.columns
]

missing_required_record_rows = 0

if required_for_valid_record:
    missing_required_mask = data[required_for_valid_record].isna().any(axis=1)
    missing_required_record_rows = int(missing_required_mask.sum())

    quarantine = pd.concat(
        [quarantine, data.loc[missing_required_mask]],
        ignore_index=True,
    )
    data = data.loc[~missing_required_mask].copy()

print(f"Invalid timestamp rows quarantined: {invalid_timestamp_rows:,}")
print(
    "Rows missing required entity or timestamp values quarantined: "
    f"{missing_required_record_rows:,}"
)
print(f"Rows remaining: {len(data):,}")
print(f"Rows quarantined: {len(quarantine):,}")

In [ ]:
# Apply configured missing-value rules.
# Adjust these rules if data_onboard specifies a different policy.
for column in NUMERIC_COLUMNS:
    if column in data.columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")
        data[column] = data[column].fillna(data[column].median())

for column in BOOLEAN_COLUMNS:
    if column in data.columns:
        data[column] = (
            data[column]
            .astype("string")
            .str.strip()
            .str.lower()
            .map(
                {
                    "true": True,
                    "1": True,
                    "yes": True,
                    "false": False,
                    "0": False,
                    "no": False,
                }
            )
        )

missing_profile_after = (
    data.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
    .assign(
        missing_fraction=lambda frame: frame["missing_count"] / max(len(data), 1)
    )
    .sort_values("missing_count", ascending=False)
)

display(missing_profile_after)

## 7. Transform Data Types and Features

In [ ]:
if TIMESTAMP_COLUMN in data.columns:
    data[TIMESTAMP_COLUMN] = pd.to_datetime(
        data[TIMESTAMP_COLUMN],
        errors="raise",
        utc=True,
    )

for column in NUMERIC_COLUMNS:
    if column in data.columns:
        data[column] = pd.to_numeric(data[column], errors="raise")

for column in BOOLEAN_COLUMNS:
    if column in data.columns:
        data[column] = data[column].astype("boolean")

for column in CATEGORICAL_COLUMNS:
    if column in data.columns:
        data[column] = data[column].astype("category")

if ENTITY_COLUMN in data.columns:
    data[ENTITY_COLUMN] = data[ENTITY_COLUMN].astype("string")

# Add forecasting-oriented calendar features when a timestamp is available.
if TIMESTAMP_COLUMN in data.columns:
    data["timestamp_year"] = data[TIMESTAMP_COLUMN].dt.year
    data["timestamp_month"] = data[TIMESTAMP_COLUMN].dt.month
    data["timestamp_day"] = data[TIMESTAMP_COLUMN].dt.day
    data["timestamp_day_of_week"] = data[TIMESTAMP_COLUMN].dt.dayofweek

display(data.head())
display(data.dtypes.rename("dtype").to_frame())

## 8. Create Training, Validation, and Test Splits

In [ ]:
fraction_total = TRAIN_FRACTION + VALIDATION_FRACTION + TEST_FRACTION

if not np.isclose(fraction_total, 1.0):
    raise ValueError("Train, validation, and test fractions must sum to 1.")

if TIMESTAMP_COLUMN not in data.columns:
    raise ValueError(
        "A timestamp column is required for chronological time-series splits."
    )

data = data.sort_values(TIMESTAMP_COLUMN).reset_index(drop=True)

unique_timestamps = data[TIMESTAMP_COLUMN].drop_duplicates().sort_values()

train_cut_index = int(len(unique_timestamps) * TRAIN_FRACTION)
validation_cut_index = int(
    len(unique_timestamps) * (TRAIN_FRACTION + VALIDATION_FRACTION)
)

train_cut_index = max(1, min(train_cut_index, len(unique_timestamps)))
validation_cut_index = max(
    train_cut_index,
    min(validation_cut_index, len(unique_timestamps)),
)

train_end = unique_timestamps.iloc[train_cut_index - 1]
validation_end = unique_timestamps.iloc[validation_cut_index - 1]

train = data.loc[data[TIMESTAMP_COLUMN] <= train_end].copy()
validation = data.loc[
    data[TIMESTAMP_COLUMN].gt(train_end)
    & data[TIMESTAMP_COLUMN].le(validation_end)
].copy()
test = data.loc[data[TIMESTAMP_COLUMN].gt(validation_end)].copy()

splits = {
    "train": train,
    "validation": validation,
    "test": test,
}

split_summary = []

for split_name, frame in splits.items():
    entity_count = (
        frame[ENTITY_COLUMN].nunique()
        if ENTITY_COLUMN in frame.columns
        else None
    )

    split_summary.append(
        {
            "split": split_name,
            "rows": len(frame),
            "entities": entity_count,
            "start": frame[TIMESTAMP_COLUMN].min(),
            "end": frame[TIMESTAMP_COLUMN].max(),
        }
    )

split_summary = pd.DataFrame(split_summary)
display(split_summary)

In [ ]:
if len(train) == 0 or len(validation) == 0 or len(test) == 0:
    raise ValueError(
        "One or more chronological splits are empty. "
        "Adjust the split fractions or provide more data."
    )

if train[TIMESTAMP_COLUMN].max() >= validation[TIMESTAMP_COLUMN].min():
    raise AssertionError("Training data overlaps validation data.")

if validation[TIMESTAMP_COLUMN].max() >= test[TIMESTAMP_COLUMN].min():
    raise AssertionError("Validation data overlaps test data.")

## 9. Save Processed Data and Metadata

In [ ]:
run_timestamp = datetime.now(timezone.utc).isoformat()

output_paths = {}

for split_name, frame in splits.items():
    output_path = OUTPUT_ROOT / f"{DATASET_NAME.lower()}_{split_name}.parquet"
    frame.to_parquet(output_path, index=False)
    output_paths[split_name] = str(output_path)

if not quarantine.empty:
    quarantine_path = OUTPUT_ROOT / f"{DATASET_NAME.lower()}_quarantine.parquet"
    quarantine.to_parquet(quarantine_path, index=False)
    output_paths["quarantine"] = str(quarantine_path)

metadata = {
    "dataset_name": DATASET_NAME,
    "processed_at_utc": run_timestamp,
    "configuration": CONFIG,
    "output_paths": output_paths,
    "source_files": [str(path) for path in discovered_files],
    "schema": {
        column: str(dtype)
        for column, dtype in data.dtypes.items()
    },
    "columns": list(data.columns),
    "row_counts": {
        split_name: int(len(frame))
        for split_name, frame in splits.items()
    },
    "quarantine_rows": int(len(quarantine)),
    "validation_results": validation_results,
}

metadata_path = OUTPUT_ROOT / f"{DATASET_NAME.lower()}_metadata.json"

with metadata_path.open("w", encoding="utf-8") as metadata_file:
    json.dump(metadata, metadata_file, indent=2, default=str)

output_paths["metadata"] = str(metadata_path)

display(pd.Series(output_paths, name="output_path"))
print(f"Processed data written to {OUTPUT_ROOT}")

## 10. Add Reproducibility and Validation Checks

In [ ]:
# Preserve the exact configuration used for this run.
configuration_path = OUTPUT_ROOT / f"{DATASET_NAME.lower()}_configuration.json"

with configuration_path.open("w", encoding="utf-8") as configuration_file:
    json.dump(
        {
            **CONFIG,
            "resolved_entity_column": ENTITY_COLUMN,
            "resolved_timestamp_column": TIMESTAMP_COLUMN,
            "resolved_target_column": TARGET_COLUMN,
            "processed_at_utc": run_timestamp,
        },
        configuration_file,
        indent=2,
        default=str,
    )

required_output_columns = set(REQUIRED_COLUMNS)

for split_name, frame in splits.items():
    if frame.empty:
        raise AssertionError(f"{split_name} output is empty.")

    missing_columns = required_output_columns.difference(frame.columns)
    if missing_columns:
        raise AssertionError(
            f"{split_name} is missing columns: {sorted(missing_columns)}"
        )

    if TIMESTAMP_COLUMN in frame.columns:
        if frame[TIMESTAMP_COLUMN].isna().any():
            raise AssertionError(
                f"{split_name} contains invalid timestamps."
            )

    key_columns = [
        column
        for column in [ENTITY_COLUMN, TIMESTAMP_COLUMN]
        if column and column in frame.columns
    ]

    if key_columns and frame.duplicated(subset=key_columns).any():
        raise AssertionError(
            f"{split_name} contains duplicate keys: {key_columns}"
        )

unexpected_missing_columns = [
    column
    for column in data.columns
    if data[column].isna().any()
    and column not in CATEGORICAL_COLUMNS
    and column not in BOOLEAN_COLUMNS
]

if unexpected_missing_columns:
    print(
        "Review remaining missing values in: "
        f"{unexpected_missing_columns}"
    )

final_summary = pd.DataFrame(
    {
        "metric": [
            "source_files",
            "processed_rows",
            "training_rows",
            "validation_rows",
            "test_rows",
            "quarantine_rows",
            "output_directory",
            "configuration_file",
        ],
        "value": [
            len(discovered_files),
            len(data),
            len(train),
            len(validation),
            len(test),
            len(quarantine),
            str(OUTPUT_ROOT),
            str(configuration_path),
        ],
    }
)

display(final_summary)
print("Data onboarding completed successfully.")